# Five radar pit sites: SNOWPACK, pits, and radar

Load all ten WY2020–WY2026 runs and compare their **WY2021** depth and SWE with the 2021 observations. Raw and temperature-corrected HRRR-AK use the same dates and glacier boundary configuration. Run with the **JIF snow modeling** kernel. Source data and model outputs are read only; no figures or diagnostic files are saved automatically.

The pits describe seasonal snow above firn. Model `HS_mod` and `SWE` describe the entire modeled column. The solid depth curves subtract each year's minimum in the preceding **August 1–October 31** window, matching the exploratory TKG4 approach. This fixed-depth approximation cannot track older firn compaction or melt and is not a layer-resolved seasonal snow estimate. No corresponding SWE baseline is subtracted.

Pit dates come from the archived original sheets, including TKG4's July 7 depth/dig date and July 8 SWE/density-sampling date (paper GT July 13). TKG3 retains its recorded date discrepancy. Radar uses July 13 for TKG3 and TKG4 and July 15 for C161, from the user-supplied Table 1 dates. TKG2 and DG1 have no Table 1 radar dates: their comparisons use the July 9–15 campaign window, with a horizontal bar instead of implying a known acquisition day. Observation times/timezones are unrecorded; UTC model day/window summaries are descriptive.

Radar selection is a **500 m horizontal radius around each pit**, with sensitivity from 100–2,000 m. Percentiles describe spatial samples, not uncertainty or independent replicates. Radar SWE uses prescribed density and is not independent of depth. Datums are provisional; TKG4 pit and model coordinates/elevations differ. The forcing notebook documents terrain assumptions.

Raw TKG4 and raw DG1 use the documented [empty-column numerical guard](numerics/README.md) for an installed SNOWPACK 3.7.0 vapor-transport crash. The other runs use the original executable directly. Keep this distinction in the run audit when interpreting results.


## Figure settings

Edit these values, then rerun this cell and the plotting cells below (or Run All). The setup cell keeps these notebook overrides; changing colors or line widths does not require editing the shared visualization module.

`MATCH_Y_AXES` matches limits across sites within each figure (depth and SWE are matched separately). `TOTAL_DEPTH_ALPHA` controls the full snow/firn depth curves. Sites are plotted from lowest to highest configured model elevation. WY2021 labels show every other month beginning October 1, with dates visible on every subplot.

In [ ]:
# Editable plotting settings for every figure in this notebook.
colors = {
    'hrrrak': 'maroon',
    'hrrrak_lapserate': 'royalblue',
}
SIMULATION_LINEWIDTH = 2.2  # Full modeled snow/firn column; also used for SWE.
ADJUSTED_LINEWIDTH = 2.2    # Solid autumn-minimum-adjusted seasonal depth.
TOTAL_DEPTH_ALPHA = 0.5    # Opacity of every total snow/firn depth curve.
MATCH_Y_AXES = True       # Match limits within each figure; depth/SWE stay separate.


In [ ]:
from pathlib import Path
import sys
import json
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'sites/radar_sites/prepare_radar_site_forcing.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open from inside jif_snow_modeling')
sys.path.insert(0, str(ROOT/'sites/radar_sites'))
from visualization import (SIMULATION_LABELS as labels, OBSERVATION_COLOR,
    OBSERVATION_MARKERS, RC_PARAMS, comparison_legend)
from matplotlib.lines import Line2D
plt.rcParams.update(RC_PARAMS)
# Refresh dependencies so an older kernel cannot retain the four-site list.
import prepare_radar_site_forcing as forcing_module
import compare_model_pit_radar as comparison_module
importlib.reload(forcing_module)
importlib.reload(comparison_module)
load_comparison = comparison_module.load_comparison
FORCINGS = comparison_module.FORCINGS
RADAR_WINDOWS = comparison_module.RADAR_WINDOWS
RADIUS_M = 500
result = load_comparison(radius=RADIUS_M)
pits=result['pits'];models=result['models'];adjusted=result['adjusted'];local=result['local_radar']
# Derive plotted sites from the loaded runs and check none were omitted.
SITES = tuple(dict.fromkeys(site for site, forcing in models))
if set(SITES) != set(forcing_module.SITES):
    raise RuntimeError('Loaded model sites do not match the configured sites; review the run audit.')
# Row-major subplot order follows configured model elevation, low to high.
from prepare_radar_site_forcing import MODEL_ROOT, PERIOD
elevations = {}
for site in SITES:
    metadata_path = MODEL_ROOT/site/'input'/f'hrrrak_{PERIOD}'/f'hrrrak_{site}_{PERIOD}.json'
    elevations[site] = json.loads(metadata_path.read_text())['elevation_m']
SITES = tuple(sorted(SITES, key=lambda site: (elevations[site], site)))
print(f'Loaded {len(SITES)} sites; subplot order: '+ ' → '.join(site.upper() for site in SITES))
display(pd.Series({site.upper(): elevations[site] for site in SITES}, name='Model elevation (m)'))
display(result['run_audit'])
display(pits[['pit_date','pit_swe_date','elevation_m','depth_cm','swe_mm','quality_flags']])
display(result['radius_sensitivity'].round(2))


## Full model record

Daily means of the total modeled column show whether snow/firn carries over between seasons. The shaded period is WY2021. These full-column curves should not be interpreted as seasonal pit depth.

In [ ]:
rows=(len(SITES)+1)//2
fig,axes=plt.subplots(rows,2,figsize=(16,4*rows),sharex=True,sharey=MATCH_Y_AXES)
for ax,site in zip(axes.flat,SITES):
    for forcing in FORCINGS:
        frame=models[(site,forcing)]
        ax.plot(frame.depth_cm.resample('D').mean(),color=colors[forcing],lw=SIMULATION_LINEWIDTH,ls='--',alpha=TOTAL_DEPTH_ALPHA,label=labels[forcing])
    ax.axvspan(pd.Timestamp('2020-10-01'),pd.Timestamp('2021-10-01'),color='0.6',alpha=.18)
    ax.set(title=site.upper(),ylabel='snow depth (cm)',xlabel='')
    ax.grid(alpha=.2)
for ax in axes.flat[:len(SITES)]: ax.tick_params(axis='x',labelbottom=True)
for ax in axes.flat[len(SITES):]: ax.set_visible(False)
for ax in axes.flat[:len(SITES)] if axes.shape[0] != len(SITES) else axes.flat:
    if ax.get_visible(): ax.tick_params(axis='y',labelleft=True)
fig.tight_layout(rect=(0,.09,1,1))
handles=[Line2D([],[],color=colors[f],lw=SIMULATION_LINEWIDTH,ls='--',alpha=TOTAL_DEPTH_ALPHA,label=labels[f]) for f in FORCINGS]
comparison_legend(fig,handles,columns=2,bottom=.10)
plt.show()


## Full model record · total and autumn-minimum-adjusted depth

The same full simulation period, now with solid autumn-minimum-adjusted depth overlaid on the faded dashed total snow/firn column. Daily means are shown for both forcing variants at every site. Solid curves break at water-year boundaries because each year has a different autumn reference. The first reference window is partial, as reported in the baseline audit. This fixed-baseline seasonal-depth approximation cannot track older firn compaction or melt. WY2021 remains shaded.

In [ ]:
rows=(len(SITES)+1)//2
fig_full_adjusted,full_adjusted_axes=plt.subplots(
    rows,2,figsize=(16,4*rows),sharex=True,sharey=MATCH_Y_AXES)
for ax,site in zip(full_adjusted_axes.flat,SITES):
    for forcing in FORCINGS:
        frame=adjusted[(site,forcing)]
        total=frame.depth_cm.resample('D').mean()
        seasonal=frame.depth_above_minimum_cm.resample('D').mean()
        water_year=pd.Series(seasonal.index.year+(seasonal.index.month>=10).astype(int),index=seasonal.index)
        # Avoid connecting different annual reference depths across October 1.
        seasonal=seasonal.mask(water_year.ne(water_year.shift()))
        ax.plot(total.index,total,color=colors[forcing],lw=SIMULATION_LINEWIDTH,
                ls='--',alpha=TOTAL_DEPTH_ALPHA,zorder=2)
        ax.plot(seasonal.index,seasonal,color=colors[forcing],lw=ADJUSTED_LINEWIDTH,
                ls='-',zorder=3)
    ax.axvspan(pd.Timestamp('2020-10-01'),pd.Timestamp('2021-10-01'),color='0.6',alpha=.18)
    ax.set(title=site.upper(),ylabel='snow depth (cm)',xlabel='')
    ax.tick_params(axis='both',labelbottom=True,labelleft=True)
    ax.grid(alpha=.2)
for ax in full_adjusted_axes.flat[len(SITES):]: ax.set_visible(False)
fig_full_adjusted.tight_layout(rect=(0,.11,1,1))
handles=[Line2D([],[],color=colors[f],lw=ADJUSTED_LINEWIDTH,
                label=labels[f]+' · seasonal depth') for f in FORCINGS]
handles += [Line2D([],[],color=colors[f],lw=SIMULATION_LINEWIDTH,ls='--',
                   alpha=TOTAL_DEPTH_ALPHA,label=labels[f]+' · total snow/firn') for f in FORCINGS]
comparison_legend(fig_full_adjusted,handles,columns=2,bottom=.12)
plt.show()


## WY2021 depth and SWE

Faded dashed depth curves show the full modeled snow/firn column. Solid depth curves show the autumn-minimum seasonal-depth approximation; SWE remains a solid total-column curve. Negative adjusted depths remain visible. Black circles show the original pit observations. Black squares show radar medians; vertical bars are spatial 10th–90th percentiles, and horizontal bars identify unresolved acquisition windows. Model time series remain visible through the window rather than selecting an arbitrary radar date.

In [ ]:
fig,axes=plt.subplots(len(SITES),2,figsize=(16,3.7*len(SITES)),sharex=True,sharey='col' if MATCH_Y_AXES else False)
for i,site in enumerate(SITES):
    pit=pits.loc[site]
    for forcing in FORCINGS:
        frame=adjusted[(site,forcing)].loc['2020-10-01':'2021-09-30']
        axes[i,0].plot(frame.index,frame.depth_cm,color=colors[forcing],lw=SIMULATION_LINEWIDTH,ls='--',alpha=TOTAL_DEPTH_ALPHA,label=labels[forcing]+' · total snow/firn')
        axes[i,0].plot(frame.index,frame.depth_above_minimum_cm,color=colors[forcing],ls='-',lw=ADJUSTED_LINEWIDTH,label=labels[forcing]+' · minus autumn minimum')
        axes[i,1].plot(frame.index,frame.swe_mm,color=colors[forcing],lw=SIMULATION_LINEWIDTH,label=labels[forcing])
    for j,(column,scale,pit_value,label) in enumerate([
        ('aa_avg_depth_m',100,pit.depth_cm,'snow depth (cm)'),
        ('aa_swe_avg_m',1000,pit.swe_mm,'SWE (mm)')]):
        ax=axes[i,j]
        ax.scatter(pit.pit_date if j==0 else pit.pit_swe_date,pit_value,c=OBSERVATION_COLOR,marker=OBSERVATION_MARKERS['pit'],s=75,zorder=6,label='Archived pit')
        samples=local[site]
        if len(samples):
            q=samples[column].quantile([.1,.5,.9])*scale
            start,end=map(pd.Timestamp,RADAR_WINDOWS[site])
            # Known radar days are plotted at noon; TKG2 and DG1 span the campaign.
            midpoint=start+(end-start)/2
            xerr=np.array([[(end-start)/pd.Timedelta(days=2)]]) if site in {'tkg2','dg1'} else None
            ax.errorbar(mdates.date2num(midpoint),q.loc[.5],yerr=[[q.loc[.5]-q.loc[.1]],[q.loc[.9]-q.loc[.5]]],
                        xerr=xerr,fmt=OBSERVATION_MARKERS['radar'],color=OBSERVATION_COLOR,capsize=3,zorder=5,label=f'Radar within {RADIUS_M} m')
        ax.set(title=f'{site.upper()} · '+('depth' if j==0 else 'SWE'),ylabel=label,
               xlim=(pd.Timestamp('2020-10-01'),pd.Timestamp('2021-10-01')))
        ax.tick_params(axis='x',labelbottom=True)
        ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[10,12,2,4,6,8],bymonthday=1));ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
        ax.grid(alpha=.2)
for ax in axes.flat[:len(SITES)] if axes.shape[0] != len(SITES) else axes.flat:
    if ax.get_visible(): ax.tick_params(axis='y',labelleft=True)
fig.tight_layout(rect=(0,.075,1,1))
handles=[Line2D([],[],color=colors[f],lw=ADJUSTED_LINEWIDTH,label=labels[f]+' · seasonal depth / total SWE') for f in FORCINGS]
handles += [Line2D([],[],color=colors[f],lw=SIMULATION_LINEWIDTH,ls='--',alpha=TOTAL_DEPTH_ALPHA,
                   label=labels[f]+' · total snow/firn') for f in FORCINGS]
handles += [Line2D([],[],color=OBSERVATION_COLOR,marker=OBSERVATION_MARKERS['pit'],ls='',markersize=8,label='Snow pit'),
            Line2D([],[],color=OBSERVATION_COLOR,marker=OBSERVATION_MARKERS['radar'],ls='',markersize=8,label=f'Radar within {RADIUS_M} m')]
comparison_legend(fig,handles,columns=3,bottom=.085)
plt.show()


## WY2021 snow depth · presentation layout

Five pit locations in a **3 × 2** layout on a 16:9 canvas; the unused sixth panel is hidden. Solid lines emphasize seasonal depth estimated by subtracting the autumn minimum. The full modeled snow/firn column is dashed and faded for context. This remains a fixed-baseline approximation, not a layer-resolved seasonal snow calculation. Black circles are pit depth and black squares are radar. SWE is omitted.

In [ ]:
# Depth-only layout for presentation. No figures are exported automatically.
fig_depth,depth_axes=plt.subplots(3,2,figsize=(18,10.125),sharex=True,sharey=MATCH_Y_AXES)
for ax,site in zip(depth_axes.flat,SITES):
    pit=pits.loc[site]
    for forcing in FORCINGS:
        frame=adjusted[(site,forcing)].loc['2020-10-01':'2021-09-30']
        ax.plot(frame.index,frame.depth_cm,color=colors[forcing],lw=SIMULATION_LINEWIDTH,
                ls='--',alpha=TOTAL_DEPTH_ALPHA,zorder=2)
        ax.plot(frame.index,frame.depth_above_minimum_cm,color=colors[forcing],
                lw=ADJUSTED_LINEWIDTH,ls='-',zorder=3)
    ax.scatter(pit.pit_date,pit.depth_cm,c=OBSERVATION_COLOR,
               marker=OBSERVATION_MARKERS['pit'],s=75,zorder=6)
    samples=local[site]
    if len(samples):
        q=samples.aa_avg_depth_m.quantile([.1,.5,.9])*100
        start,end=map(pd.Timestamp,RADAR_WINDOWS[site])
        midpoint=start+(end-start)/2
        xerr=np.array([[(end-start)/pd.Timedelta(days=2)]]) if site in {'tkg2','dg1'} else None
        ax.errorbar(mdates.date2num(midpoint),q.loc[.5],
                    yerr=[[q.loc[.5]-q.loc[.1]],[q.loc[.9]-q.loc[.5]]],xerr=xerr,
                    fmt=OBSERVATION_MARKERS['radar'],color=OBSERVATION_COLOR,capsize=3,zorder=5)
    ax.set(title=site.upper(),ylabel='snow depth (cm)',xlabel='',
           xlim=(pd.Timestamp('2020-10-01'),pd.Timestamp('2021-10-01')))
    ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[10,12,2,4,6,8],bymonthday=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%Y'))
    ax.tick_params(axis='x',labelbottom=True)
    ax.grid(alpha=.2)
for ax in depth_axes.flat[len(SITES):]: ax.set_visible(False)
for ax in depth_axes.flat[:len(SITES)]: ax.tick_params(axis='y',labelleft=True)
fig_depth.tight_layout(rect=(0,.12,1,1))
handles=[Line2D([],[],color=colors[f],lw=ADJUSTED_LINEWIDTH,label=labels[f]+' · seasonal depth') for f in FORCINGS]
handles += [Line2D([],[],color=colors[f],lw=SIMULATION_LINEWIDTH,ls='--',alpha=TOTAL_DEPTH_ALPHA,
                   label=labels[f]+' · total snow/firn') for f in FORCINGS]
handles += [Line2D([],[],color=OBSERVATION_COLOR,marker=OBSERVATION_MARKERS['pit'],ls='',markersize=8,label='Snow pit'),
            Line2D([],[],color=OBSERVATION_COLOR,marker=OBSERVATION_MARKERS['radar'],ls='',markersize=8,label=f'Radar within {RADIUS_M} m')]
comparison_legend(fig_depth,handles,columns=3,bottom=.15)
fig_depth.subplots_adjust(hspace=.65)
plt.show()


## Observation windows and baseline audit

Model means are reported only for complete observation windows. Minima/maxima show temporal variation, while radar percentile columns describe spatial variation. These are descriptive comparisons of different column definitions. The first autumn baseline is partial because the simulations start in October 2019. Review the radius sensitivity and exact radar dates before interpreting apparent model agreement.

In [ ]:
display(result['comparisons'].round(2))
display(result['baselines'].round(2))


Reusable code is in `compare_model_pit_radar.py`; it can also be run as a script to print the run audit and observation comparisons. Future work should extract the 2020/21 horizon from model layer profiles before treating pit/radar comparisons as seasonal-snow validation.

## Snow-depth summary statistics

Compare the **autumn-minimum-adjusted model depth** (solid lines) with the archived pit depth and the median radar depth within 500 m. Positive model-minus-observation values mean the model is deeper; negative values mean it is shallower. All depths and differences are in **cm**.

Model depth is averaged over the pit's collection day for the pit comparison, and over the radar day/window for the radar comparison. TKG2 and DG1 retain the July 9–15 campaign window because their exact radar dates are unresolved. Therefore separate model values are shown for the two observation windows.

Mean absolute difference (MAE) summarizes the magnitude without allowing overestimates and underestimates to cancel. Mean signed difference (bias) retains their direction. Each site/forcing/source comparison receives equal weight, regardless of the number of radar samples. These are descriptive discrepancies for a fixed-baseline seasonal-depth approximation; radar spatial spread is not an uncertainty bound on model error.

In [ ]:
importlib.reload(comparison_module)
stats = comparison_module.snow_depth_statistics(result['comparisons'])
depth_summary = stats['detail'].copy()
depth_summary['site'] = pd.Categorical(depth_summary.site,categories=SITES,ordered=True)
depth_summary = depth_summary.sort_values(['site','forcing']).reset_index(drop=True)
depth_summary['simulation'] = depth_summary.forcing.map(labels)
depth_summary['radar_window'] = depth_summary.apply(
    lambda r: r.radar_window_start.strftime('%Y-%m-%d') if
    r.radar_window_end_exclusive-r.radar_window_start == pd.Timedelta(days=1) else
    r.radar_window_start.strftime('%Y-%m-%d')+'–'+
    (r.radar_window_end_exclusive-pd.Timedelta(days=1)).strftime('%Y-%m-%d'),axis=1)
columns = ['site','simulation','pit_date','pit_depth_cm','model_at_pit_cm','model_minus_pit_cm',
           'radar_window','radar_median_depth_cm','model_at_radar_cm','model_minus_radar_cm']
summary_display = depth_summary[columns].rename(columns={
    'site':'Site','simulation':'Simulation','pit_date':'Pit date','pit_depth_cm':'Pit (cm)',
    'model_at_pit_cm':'Model at pit date (cm)','model_minus_pit_cm':'Model − pit (cm)',
    'radar_window':'Radar date/window','radar_median_depth_cm':'Radar median (cm)',
    'model_at_radar_cm':'Model at radar window (cm)','model_minus_radar_cm':'Model − radar (cm)'})
summary_display['Site'] = summary_display['Site'].astype(str).str.upper()
formats={column:'{:.1f}' for column in summary_display.select_dtypes(include='number').columns}
formats.update({'Model − pit (cm)':'{:+.1f}','Model − radar (cm)':'{:+.1f}',
                'Pit date':lambda t:t.strftime('%Y-%m-%d')})
display(summary_display.style.format(formats).hide(axis='index'))

by_simulation = stats['by_simulation'].reset_index()
by_simulation['forcing'] = by_simulation.forcing.map(labels)
by_simulation['observation'] = by_simulation.observation.replace({'Pit depth':'Snow pit'})
by_simulation = by_simulation.rename(columns={'forcing':'Simulation','observation':'Observation',
    'comparisons':'Site comparisons','mean_bias_cm':'Mean bias (cm)',
    'mean_absolute_difference_cm':'Mean absolute difference (cm)'})
display(by_simulation.style.format({'Mean bias (cm)':'{:+.1f}',
    'Mean absolute difference (cm)':'{:.1f}'}).hide(axis='index'))


In [ ]:
from IPython.display import Markdown
pit_mae = stats['by_source'].loc['Pit depth','mean_absolute_difference_cm']
radar_mae = stats['by_source'].loc['Radar','mean_absolute_difference_cm']
overall_mae = stats['overall_mean_absolute_difference_cm']
display(Markdown(
    f"**Overall mean absolute model–observation depth difference: {overall_mae:.1f} cm.**\n\n"
    f"Across both forcing variants and all five sites, the mean absolute difference is "
    f"**{pit_mae:.1f} cm from snow pits** and **{radar_mae:.1f} cm from radar medians**. "
    f"The overall value gives equal weight to the {stats['comparisons']} site/forcing/source pairs; "
    "it is a discrepancy magnitude, not a ± confidence interval."))
